In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import requests
import os

import  tensorflow as tf
from tensorflow.keras import models,layers,optimizers,regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D,GlobalAveragePooling2D, Dense, Flatten,Embedding,Dropout
from tensorflow.keras.applications import EfficientNetB0

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

In [3]:
data = pd.read_csv('FINAL_DATASET.csv')
data.head()

,image_id,image_url,label,label_numeric,category,gender,age_group,source,fake_method,image_quality,resolution,confidence_score,detection_difficulty,dataset_split,date_collected,version,year
0,1,https://images.unsplash.com/photo-172422561835...,REAL,1,Authentic,Female,18-25,Unsplash,NaN,High,1080x1080,0.96,Easy,train,2026-04-24,v2.0,2026
1,2,https://images.unsplash.com/photo-172422561873...,REAL,1,Authentic,Male,50+,Unsplash,NaN,High,1080x1080,0.97,Easy,train,2026-04-04,v2.0,2026
2,3,https://images.unsplash.com/photo-172422561834...,REAL,1,Authentic,Female,36-50,Unsplash,NaN,High,1080x1080,0.95,Easy,test,2026-02-07,v2.0,2026
3,4,https://images.unsplash.com/photo-172422561855...,REAL,1,Authentic,Female,26-35,Unsplash,NaN,High,1080x1080,0.89,Easy,train,2026-02-20,v2.0,2026
4,5,https://images.unsplash.com/photo-172422561823...,REAL,1,Authentic,Male,36-50,Unsplash,NaN,High,1080x1080,0.86,Easy,train,2026-04-12,v2.0,2026


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6557 entries, 0 to 6556
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   image_id              6557 non-null   int64  
 1   image_url             6557 non-null   object 
 2   label                 6557 non-null   object 
 3   label_numeric         6557 non-null   int64  
 4   category              6557 non-null   object 
 5   gender                6557 non-null   object 
 6   age_group             6557 non-null   object 
 7   source                6557 non-null   object 
 8   fake_method           3767 non-null   object 
 9   image_quality         6557 non-null   object 
 10  resolution            6557 non-null   object 
 11  confidence_score      6557 non-null   float64
 12  detection_difficulty  6557 non-null   object 
 13  dataset_split         6557 non-null   object 
 14  date_collected        6557 non-null   object 
 15  version              

In [ ]:
# import pandas as pd
# import requests
# import os
# from PIL import Image
# from io import BytesIO

# df = pd.read_csv("FINAL_DATASET.csv")

# os.makedirs("images", exist_ok=True)

# valid_rows = []

# for _, row in df.iterrows():

#     try:
#         response = requests.get(
#             row["image_url"],
#             timeout=10
#         )

#         if response.status_code != 200:
#             continue

#         if "image" not in response.headers.get(
#             "Content-Type", ""
#         ):
#             continue

#         img = Image.open(
#             BytesIO(response.content)
#         )

#         img.verify()

#         filename = f"images/{row['image_id']}.jpg"

#         with open(filename, "wb") as f:
#             f.write(response.content)

#         valid_rows.append(row)

#     except:
#         continue

# clean_df = pd.DataFrame(valid_rows)

# clean_df["filepath"] = (
#     "images/" +
#     clean_df["image_id"].astype(str) +
#     ".jpg"
# )

# clean_df.to_csv(
#     "clean_dataset.csv",
#     index=False
# )

data["filepath"] = "images/" + data["image_id"].astype(str)+'.jpg'
print(data[["filepath","label"]].head())

       filepath label
0  images/1.jpg  REAL
1  images/2.jpg  REAL
2  images/3.jpg  REAL
3  images/4.jpg  REAL
4  images/5.jpg  REAL


In [14]:
data = data[data["filepath"].apply(os.path.exists)]
len(data)

5929

In [36]:
valid_rows = []
for _,row in data.iterrows():
    try:
        Image.open(row['filepath']).verify()
        valid_rows.append(row)
    except:
        pass
clean_data = pd.DataFrame(valid_rows)

In [38]:
clean_data.to_csv("clean_data.csv",index = False)

In [39]:
data = pd.read_csv("clean_data.csv")

In [40]:
train_data,test_data = train_test_split(data,test_size = 0.2,stratify=data["label"],random_state=42)
val_data,test_data = train_test_split(data,test_size = 0.2,stratify=data["label"],random_state=42)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

image_size = 224
batch_size = 32

train_datagen = ImageDataGenerator(rescale = 1./255,
                                   rotation_range = 20,
                                   zoom_range = 0.2,
                                   horizontal_flip = True)
test_datagen = ImageDataGenerator(rescale = 1./255)

train_generator = train_datagen.flow_from_dataframe(dataframe = train_data,
                                                    x_col = "filepath",
                                                    y_col="label",
                                                    target_size = (image_size,image_size),
                                                    batch_size = batch_size,
                                                    class_mode = "binary")
test_generator = test_datagen.flow_from_dataframe(dataframe = test_data,
                                                  x_col = "filepath",
                                                  y_col = "label",
                                                  target_size = (image_size,image_size),
                                                  batch_size = batch_size,
                                                  class_mode = "binary")
val_generator = test_datagen.flow_from_dataframe(dataframe = val_data,
                                                  x_col = "filepath",
                                                y_col = "label",
                                                  target_size = (image_size,image_size),
                                                  batch_size = batch_size,
                                                  class_mode = "binary")

Found 3943 validated image filenames belonging to 2 classes.
Found 986 validated image filenames belonging to 2 classes.
Found 3943 validated image filenames belonging to 2 classes.


In [42]:
base_model = EfficientNetB0(
    weights = "imagenet",
    include_top = False,
    input_shape = (224,224,3)
)

In [43]:
base_model.trainable = False
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256,activation = "relu"),
    Dropout(0.5),
    Dense(128,activation = "relu"),
    Dropout(0.3),
    Dense(1,activation = "sigmoid")
    
])

model.compile(optimizer = "adam",loss = "binary_crossentropy",metrics = ["accuracy"])

In [44]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,410,532 (16.82 MB)

 Trainable params: 360,961 (1.38 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [45]:
history = model.fit(train_generator,validation_data = val_generator,epochs = 10)

Epoch 1/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 257s 2s/step - accuracy: 0.5184 - loss: 0.7055 - val_accuracy: 0.5298 - val_loss: 0.6916
Epoch 2/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 222s 2s/step - accuracy: 0.5044 - loss: 0.6974 - val_accuracy: 0.5298 - val_loss: 0.6924
Epoch 3/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 410s 3s/step - accuracy: 0.5191 - loss: 0.6937 - val_accuracy: 0.5298 - val_loss: 0.6918
Epoch 4/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 237s 2s/step - accuracy: 0.5298 - loss: 0.6918 - val_accuracy: 0.5298 - val_loss: 0.6915
Epoch 5/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 1106s 9s/step - accuracy: 0.5298 - loss: 0.6915 - val_accuracy: 0.5298 - val_loss: 0.6914
Epoch 6/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 227s 2s/step - accuracy: 0.5298 - loss: 0.6915 - val_accuracy: 0.5298 - val_loss: 0.6914
Epoch 7/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 325s 3s/step - accuracy: 0.5298 - loss: 0.6915 - val_accuracy: 0.5298 - val_loss: 0.6914
Epoch 8/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 255s 2s/step - accuracy: 0.5283 - loss: 0.6917 - val_acc

UnknownError: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
FileNotFoundError: [Errno 2] No such file or directory: 'images/1084.jpg'
Traceback (most recent call last):

  File "c:\Users\SIBICHAKRAVARTHI\anaconda3\Lib\site-packages\tensorflow\python\ops\script_ops.py", line 269, in __call__
    ret = func(*args)

  File "c:\Users\SIBICHAKRAVARTHI\anaconda3\Lib\site-packages\tensorflow\python\autograph\impl\api.py", line 643, in wrapper
    return func(*args, **kwargs)

  File "c:\Users\SIBICHAKRAVARTHI\anaconda3\Lib\site-packages\tensorflow\python\data\ops\from_generator_op.py", line 198, in generator_py_func
    values = next(generator_state.get_iterator(iterator_id))

  File "c:\Users\SIBICHAKRAVARTHI\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py", line 264, in _finite_generator
    yield self._standardize_batch(self.py_dataset[i])
                                  ~~~~~~~~~~~~~~~^^^

1000